In [0]:
%run ../config

In [0]:
df = spark.table(f"{catalog}.{bronze_schema}.acled_raw")
print(f' {display(df.count())} table loaded from bronze:')

In [0]:
west_africa_countries = [
    "Nigeria", "Ghana", "Senegal", "Mali", "Burkina Faso",
    "Niger", "Guinea", "Sierra Leone", "Liberia", "Ivory Coast",
    "Togo", "Benin", "Gambia", "Guinea-Bissau", "Cape Verde", "Mauritania"
]

df_west_africa = df.filter(df["country"].isin(west_africa_countries))

print(f" Filtered to West Africa")
print(f"   Rows: {df_west_africa.count()}")
print(f"   Countries: {df_west_africa.select('country').distinct().count()}")

In [0]:
display(df.select("country").distinct().orderBy("country"))

In [0]:
display(df_west_africa.select("country").distinct().orderBy("country"))

In [0]:
from pyspark.sql.functions import col

df_selected = df_west_africa.select(
    col("id").alias("event_id"),
    col("year"),
    col("date_start"),
    col("date_end"),
    col("country"),
    col("region"),
    col("adm_1").alias("state_province"),
    col("adm_2").alias("district"),
    col("latitude"),
    col("longitude"),
    col("type_of_violence"),
    col("conflict_name"),
    col("dyad_name"),
    col("side_a"),
    col("side_b"),
    col("deaths_a"),
    col("deaths_b"),
    col("deaths_civilians"),
    col("deaths_unknown"),
    col("best").alias("best_estimate_deaths"),
    col("high").alias("high_estimate_deaths"),
    col("low").alias("low_estimate_deaths"),
    col("source")
)

print(f" Columns selected and renamed")
print(f"   Rows: {df_selected.count()}")
print(f"   Columns: {len(df_selected.columns)}")

In [0]:
df_selected.display()

In [0]:
from pyspark.sql.functions import col, to_date

df_typed = df_selected \
    .withColumn("date_start", to_date(col("date_start"))) \
    .withColumn("date_end", to_date(col("date_end"))) \
    .withColumn("year", col("year").cast("integer")) \
    .withColumn("latitude", col("latitude").cast("double")) \
    .withColumn("longitude", col("longitude").cast("double")) \
    .withColumn("type_of_violence", col("type_of_violence").cast("integer")) \
    .withColumn("deaths_a", col("deaths_a").cast("integer")) \
    .withColumn("deaths_b", col("deaths_b").cast("integer")) \
    .withColumn("deaths_civilians", col("deaths_civilians").cast("integer")) \
    .withColumn("deaths_unknown", col("deaths_unknown").cast("integer")) \
    .withColumn("best_estimate_deaths", col("best_estimate_deaths").cast("integer")) \
    .withColumn("high_estimate_deaths", col("high_estimate_deaths").cast("integer")) \
    .withColumn("low_estimate_deaths", col("low_estimate_deaths").cast("integer"))

print(" Columns cast to correct types")
df_typed.printSchema()

In [0]:
df_decoded = df_typed.withColumn(
    "violence_type_desc",
    F.when(col("type_of_violence") == 1, "State-Based Violence")
    .when(col("type_of_violence") == 2, "Non-State Violence")
    .when(col("type_of_violence") == 3, "One-Sided Violence (Civilians)")
    .otherwise("Unknown")
)

print(" Violence type decoded")
display(df_decoded.select("type_of_violence", "violence_type_desc").distinct().orderBy("type_of_violence"))

In [0]:
df_clean = df_decoded \
    .fillna(0, subset=["deaths_a", "deaths_b", "deaths_civilians", "deaths_unknown", 
                        "best_estimate_deaths", "high_estimate_deaths", "low_estimate_deaths"]) \
    .fillna("Unknown", subset=["state_province", "district", "dyad_name", "side_a", "side_b"])

print(" Nulls handled")
print(f"   Rows: {df_clean.count()}")

In [0]:
(df_clean.write 
    .format("delta") 
    .mode("overwrite") 
    .option("overwriteSchema", "true") 
    .saveAsTable(f" {catalog}.{silver_schema}.events_cleaned")

)
print(f" Silver table written: {catalog}.{silver_schema}.events_cleaned")
print(f"   Total rows: {df_clean.count()}")